In [13]:
from re import S
import torch
import numpy as np
from torch import nn
from torch.autograd import grad 
from torch_geometric.data import DataLoader
from torch.utils.data import DataLoader as DataLoaders
from time import time
import os.path as osp
from torch_geometric.utils import to_dense_batch, k_hop_subgraph
from INfluence.inverse_hvp import get_inverse_hvp
#from utils.my_math import masked_mae_np, masked_mape_np, masked_mse_np
from trafficDataset import TrafficDataset
from model.model import Basic_Model
from utils.common_tools import ct

ModuleNotFoundError: No module named 'utils'

In [5]:
def sav_s_test(model,    subgraph,
                          inputs,
                          preproc_fn,
                          approx_type='lissa',
                          lissa_params='LISSA_PARAMS',
                          verbose=True):
    #为每一个节点求一个
    params = list(model.parameters())
    criterion = nn.CrossEntropyLoss()
    train_data=TrafficDataset("", "", x=inputs["train_x"][:, :, subgraph.numpy()], y=inputs["train_y"][:, :, subgraph.numpy()],edge_index="", mode="subgraph")
    test_dataset=TrafficDataset("", "", x=inputs["val_x"][:, :, subgraph.numpy()], y=inputs["val_y"][:, :, subgraph.numpy()],edge_index="", mode="subgraph")
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    s_test_values = []
    for s_test_idx, (input, target) in enumerate(test_loader):
        if verbose:
            start_time = time()
        
        input, target = preproc_fn(input, target)
        loss = criterion(model(input), target)
        test_grads = grad(loss, params)

        s_test = get_inverse_hvp(model, criterion, train_data, test_grads,
                                                approx_type='lissa',
                                                approx_params=lissa_params,
                                                preproc_data_fn=preproc_fn)
        s_test_values.append(s_test)


In [6]:
path='/home/wbw/ijcai3/ijcai2/ijcai2/res/district3F11T17/trafficStream2021-12-07-10:20:59.831263/2011/33.1277.pkl'
conf_path = 'conf/trafficStream.json'
args = ct.load_json_file(conf_path)
model.load_state_dict(torch.load(best_model_path, args.device)["model_state_dict"])

NameError: name 'ct' is not defined